---
# 🔧 FIXES & UPGRADES — Cellules Correctives
## À ajouter à la suite du notebook principal

Ces cellules corrigent **4 problèmes identifiés** dans le notebook v1 :

| Fix | Problème | Solution |
|-----|----------|----------|
| **A** | LabelEncoder crée un ordre artificiel | ✅ OneHotEncoder + Pipeline sklearn sans leakage |
| **B** | Fit sur train+test = data leakage | ✅ Fit sur train uniquement, handle_unknown sur test |
| **C** | Fitness = Accuracy seulement (pas IDS-optimal) | ✅ Fitness = λ·F1_macro + (1-λ)·Recall_attack |
| **D** | P=1.000 partout = convergence trop agressive | ✅ Repair strict k features + entropy regularization |
| **E** | `sys.getsizeof` faux sur numpy → 100% erroné | ✅ `.nbytes` pour la vraie mémoire |

---

## 🔧 FIX A+B — Prétraitement Corrigé : OneHotEncoder sans Data Leakage

### Problème A : LabelEncoder = ordre artificiel
LabelEncoder assigne des entiers arbitraires : `ftp_data=0`, `http=1`, `smtp=2`...
Le modèle interprète alors `smtp > http > ftp_data` ce qui est **faux** — ces services n'ont pas d'ordre.

**Impact sur le QIEA** : la Q-gate peut favoriser ou pénaliser une feature catégorielle
à cause d'une corrélation artificielle créée par l'encodage.

### Problème B : `le.fit(combined)` = Data Leakage
Inclure le test dans le fit expose des informations du futur au prétraitement.
Même si l'impact est faible ici, c'est une **mauvaise pratique** critiquable.

### Solution : Pipeline sklearn
```
ColumnTransformer
├── OneHotEncoder(handle_unknown='ignore')  ← cat cols, fit sur train seulement
└── RobustScaler()                          ← num cols
```
- `handle_unknown='ignore'` : si le test contient une catégorie inconnue → colonne de zéros
- Résultat : ~122 features (41 num + OHE des 3 cat) au lieu de 41
- Le QIEA sélectionne ensuite 10 parmi ces 122 → sélection **plus propre**

In [ ]:
# ═══════════════════════════════════════════════════════════
# FIX A+B : Prétraitement OneHotEncoder sans Data Leakage
# ═══════════════════════════════════════════════════════════

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

print('═' * 60)
print('  FIX A+B : OneHotEncoder + Pipeline sans Leakage')
print('═' * 60)

# ── Séparation colonnes ──────────────────────────────────────
CAT_COLS = ['protocol_type', 'service', 'flag']
NUM_COLS = [c for c in data_train.columns
            if c not in CAT_COLS + ['outcome', 'level']]

print(f'   Colonnes numériques  : {len(NUM_COLS)}')
print(f'   Colonnes catégorielles : {CAT_COLS}')

# ── Copies propres (sans modifier data_train / data_test) ────
train_fix = data_train.copy()
test_fix  = data_test.copy()

# Binarisation de la cible (identique à avant)
for df in [train_fix, test_fix]:
    df['outcome'] = df['outcome'].apply(lambda x: 0 if x == 'normal' else 1)

# ── Pipeline ColumnTransformer ───────────────────────────────
# IMPORTANT : fit UNIQUEMENT sur train → pas de leakage
preprocessor = ColumnTransformer(
    transformers=[
        # Colonnes numériques → RobustScaler
        ('num', RobustScaler(), NUM_COLS),
        # Colonnes catégorielles → OneHotEncoder
        # handle_unknown='ignore' : catégories inconnues dans test → 0
        ('cat', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False   # dense array pour sklearn
        ), CAT_COLS)
    ],
    remainder='drop'  # Supprime 'level' et 'outcome'
)

# Features X et target y
X_raw_train = train_fix[NUM_COLS + CAT_COLS]
X_raw_test  = test_fix[NUM_COLS + CAT_COLS]
y_train_fix = train_fix['outcome'].values
y_test_fix  = test_fix['outcome'].values

# fit sur train UNIQUEMENT
X_train_fix = preprocessor.fit_transform(X_raw_train)
# transform sur test (sans re-fit)
X_test_fix  = preprocessor.transform(X_raw_test)

# ── Récupération des noms de features après OHE ─────────────
ohe_feature_names = (
    preprocessor
    .named_transformers_['cat']
    .get_feature_names_out(CAT_COLS)
    .tolist()
)
FEATURE_NAMES_FIX = NUM_COLS + ohe_feature_names

print(f'\n📐 Dimensions après OneHotEncoding :')
print(f'   X_train_fix : {X_train_fix.shape}')
print(f'   X_test_fix  : {X_test_fix.shape}')
print(f'   → {len(NUM_COLS)} features numériques')
print(f'   → {len(ohe_feature_names)} features OHE ({len(CAT_COLS)} colonnes cat)')
print(f'   → {len(FEATURE_NAMES_FIX)} features TOTALES (vs 41 avant)')
print('\n   ✅ Fix A : LabelEncoder → OneHotEncoder (pas d\'ordre artificiel)')
print('   ✅ Fix B : fit sur train uniquement (pas de leakage)')

## 🔧 FIX C — Fitness IDS-Optimale : λ·F1_macro + (1-λ)·Recall_attack

### Pourquoi Accuracy est insuffisante pour un IDS ?

Rappel des résultats v1 :
- Recall attaque baseline : **61.9%** → 38% des attaques sont **manquées** !
- Accuracy élevée ≠ bon IDS si le Recall est bas

### La nouvelle fonction fitness mixte :

$$\text{Fitness} = \lambda \cdot F1_{\text{macro}} + (1-\lambda) \cdot \text{Recall}_{\text{attack}}$$

Avec $\lambda = 0.4$ :
- **60%** du score vient du Recall sur la classe attaque (priorité détection)
- **40%** vient du F1_macro (équilibre global entre les deux classes)

**Conséquence** : le QIEA va maintenant favoriser les features qui **détectent les attaques**,
pas celles qui maximisent le score global.
C'est exactement l'objectif d'un **IDS** !

In [ ]:
# ═══════════════════════════════════════════════════════════
# FIX C : QIEA avec Fitness IDS-Optimale
# ═══════════════════════════════════════════════════════════

print('═' * 60)
print('  FIX C : Fitness = λ·F1_macro + (1-λ)·Recall_attack')
print('═' * 60)

LAMBDA_FITNESS = 0.4  # 40% F1_macro + 60% Recall_attack

print(f'   λ = {LAMBDA_FITNESS} → Fitness = {LAMBDA_FITNESS}·F1_macro + {1-LAMBDA_FITNESS}·Recall_attack')
print(f'   Priorité : détecter les attaques (Recall_attack = {1-LAMBDA_FITNESS:.0%} du score)')


class QIEA_IDSOptimal:
    """
    QIEA avec fitness orientée IDS :
      Fitness = λ·F1_macro + (1-λ)·Recall_attack
    
    Et deux améliorations structurelles :
      - Repair strict : exactement k features à chaque évaluation
      - Entropy regularization : évite la convergence P=1 trop rapide
    """

    def __init__(
        self,
        n_features_target : int   = 10,
        n_generations     : int   = 30,
        population_size   : int   = 10,
        rotation_angle    : float = 0.05 * np.pi,
        n_estimators_eval : int   = 30,
        lambda_fitness    : float = 0.4,   # FIX C
        entropy_reg       : float = 0.01,  # FIX D
        verbose           : bool  = True
    ):
        self.n_features_target  = n_features_target
        self.n_generations      = n_generations
        self.population_size    = population_size
        self.rotation_angle     = rotation_angle
        self.n_estimators_eval  = n_estimators_eval
        self.lambda_fitness     = lambda_fitness
        self.entropy_reg        = entropy_reg
        self.verbose            = verbose

        self.best_features_mask  = None
        self.best_fitness        = -np.inf
        self.fitness_history     = []
        self.recall_history      = []   # nouveau : suivre le recall
        self.f1_history          = []   # nouveau : suivre le f1
        self.theta_history       = []
        self.final_probabilities = None

    # ── Initialisation ──────────────────────────────────────
    def _init_qubits(self, n):
        return np.full(n, np.pi / 4)

    # ── FIX D : Repair strict → exactement k features ────────
    def _repair(self, chromosome: np.ndarray) -> np.ndarray:
        """
        Garantit exactement n_features_target bits à 1.
        
        Règle :
          - Si trop de 1 → désactive les moins probables (rand)
          - Si trop peu de 1 → active les plus probables parmi les 0
        
        Cela évite :
          - Les chromosomes "presque vides" (pénalisés à 0)
          - Les chromosomes "trop pleins" (biais vers features faciles)
        """
        chrom = chromosome.copy()
        k     = self.n_features_target
        on    = np.where(chrom == 1)[0]
        off   = np.where(chrom == 0)[0]

        if len(on) > k:
            # Trop de features → désactive aléatoirement l'excédent
            to_off = np.random.choice(on, len(on) - k, replace=False)
            chrom[to_off] = 0

        elif len(on) < k:
            # Trop peu de features → active aléatoirement le manque
            n_needed = k - len(on)
            to_on = np.random.choice(off, min(n_needed, len(off)), replace=False)
            chrom[to_on] = 1

        return chrom

    # ── FIX C : Fitness IDS-optimale ─────────────────────────
    def _fitness(self, chromosome, X_tr, y_tr, X_val, y_val):
        """
        Fitness = λ·F1_macro + (1-λ)·Recall_attack
        
        F1_macro : équilibre les deux classes (normal + attaque)
        Recall_attack : proportion d'attaques correctement détectées
        
        Retourne aussi recall et f1 séparément pour l'historique.
        """
        idx = np.where(chromosome == 1)[0]
        if len(idx) < 3:
            return 0.0, 0.0, 0.0

        clf = RandomForestClassifier(
            n_estimators=self.n_estimators_eval,
            max_depth=5, n_jobs=-1, random_state=42
        )
        clf.fit(X_tr[:, idx], y_tr)
        y_pred = clf.predict(X_val[:, idx])

        f1_mac  = f1_score(y_val, y_pred, average='macro', zero_division=0)
        rec_atk = recall_score(y_val, y_pred, pos_label=1, zero_division=0)

        # Formule mixte
        score = self.lambda_fitness * f1_mac + (1 - self.lambda_fitness) * rec_atk
        return score, f1_mac, rec_atk

    # ── FIX D : Rotation + Entropy Regularization ────────────
    def _rotate(self, theta, current_chrom, best_chrom):
        """
        Q-gate + entropy regularization.
        
        L'entropy regularization ajoute un bruit léger pour
        éviter que tous les θ convergent vers 0 ou π/2 trop vite.
        
        Sans régularisation : P = [1,1,1,1,1,1,1,1,1,1,0,0,...]
        Avec régularisation : P = [0.98,0.95,0.92,...,0.12,0.05,...]
        → La sélection finale est plus discriminante.
        """
        new_theta = theta.copy()

        for i in range(len(theta)):
            b_c = current_chrom[i]
            b_b = best_chrom[i]

            if   b_c == 1 and b_b == 1: direction = +1
            elif b_c == 0 and b_b == 0: direction = -1
            elif b_c == 1 and b_b == 0: direction = -1
            else:                       direction = +1

            # Rotation de base
            new_theta[i] = theta[i] + direction * self.rotation_angle

            # FIX D : Entropy regularization
            # Tire vers π/4 (état de superposition maximale)
            # proportionnellement à entropy_reg
            # → Ralentit la convergence, évite P=1 partout
            new_theta[i] += self.entropy_reg * (np.pi / 4 - new_theta[i])

            new_theta[i] = np.clip(new_theta[i], 0.05, np.pi / 2 - 0.05)

        return new_theta

    # ── Boucle principale ────────────────────────────────────
    def fit(self, X_tr, y_tr, X_val, y_val, feature_names=None):
        n = X_tr.shape[1]
        self.feature_names_ = feature_names or [f'f{i}' for i in range(n)]

        if self.verbose:
            print('═' * 65)
            print('  QIEA v2 — Fitness IDS-Optimale + Repair Strict')
            print('═' * 65)
            print(f'  Features totales  : {n}')
            print(f'  Features cibles   : {self.n_features_target} (repair strict)')
            print(f'  Fitness           : {self.lambda_fitness}·F1_macro + '
                  f'{1-self.lambda_fitness}·Recall_attack')
            print(f'  Entropy reg       : {self.entropy_reg}')
            print('═' * 65)

        theta           = self._init_qubits(n)
        best_chrom      = np.zeros(n, dtype=int)
        best_fitness    = -np.inf
        t_start         = time.perf_counter()

        for gen in range(self.n_generations):
            gen_scores, gen_chroms = [], []

            for _ in range(self.population_size):
                # Observation
                chrom = (np.random.rand(n) < np.sin(theta)**2).astype(int)
                # FIX D : Repair strict → exactement k features
                chrom = self._repair(chrom)
                # FIX C : Fitness IDS-optimale
                score, f1m, rec = self._fitness(chrom, X_tr, y_tr, X_val, y_val)
                gen_scores.append((score, f1m, rec))
                gen_chroms.append(chrom)

            # Meilleur de la génération
            best_idx = np.argmax([s[0] for s in gen_scores])
            best_g   = gen_scores[best_idx][0]

            if best_g > best_fitness:
                best_fitness = best_g
                best_chrom   = gen_chroms[best_idx].copy()

            # Mise à jour Q-gate
            for chrom in gen_chroms:
                theta = self._rotate(theta, chrom, best_chrom)

            # Historique
            self.fitness_history.append(best_fitness)
            self.recall_history.append(gen_scores[best_idx][2])
            self.f1_history.append(gen_scores[best_idx][1])
            self.theta_history.append(theta.copy())

            if self.verbose and (gen % 5 == 0 or gen == self.n_generations-1):
                _, f1m, rec = gen_scores[best_idx]
                print(f'  Gén {gen+1:3d}/{self.n_generations} | '
                      f'Fitness: {best_fitness:.4f} | '
                      f'F1_macro: {f1m:.4f} | '
                      f'Recall_atk: {rec:.4f}')

        # Sélection finale sur probabilités
        final_probs = np.sin(theta)**2
        top_k       = np.argsort(final_probs)[-self.n_features_target:]
        mask        = np.zeros(n, dtype=int)
        mask[top_k] = 1

        self.best_features_mask  = mask
        self.best_fitness        = best_fitness
        self.final_probabilities = final_probs

        t_elapsed = time.perf_counter() - t_start
        if self.verbose:
            print('═' * 65)
            print(f'  ✅ QIEA v2 terminé en {t_elapsed:.1f}s')
            print(f'  🏆 Meilleur fitness (mixte) : {best_fitness:.4f}')
            names_sel = [self.feature_names_[i] for i in top_k]
            probs_sel = final_probs[top_k]
            print(f'  📌 Features sélectionnées (avec probabilités) :')
            for nm, p in sorted(zip(names_sel, probs_sel), key=lambda x: -x[1]):
                bar = '█' * int(p*20) + '░' * (20 - int(p*20))
                print(f'     {bar} {p:.3f}  {nm}')
            print('═' * 65)
        return self

    def get_selected_indices(self):
        return np.where(self.best_features_mask == 1)[0]

    def get_selected_names(self):
        return [self.feature_names_[i] for i in self.get_selected_indices()]


print('\n✅ Classe QIEA_IDSOptimal définie !')
print('   → Fix C : Fitness = λ·F1_macro + (1-λ)·Recall_attack')
print('   → Fix D : Repair strict + Entropy regularization')

## 🚀 Lancement du QIEA v2 (sur données OHE)

On applique le QIEA corrigé sur les données prétraitées avec OneHotEncoder (Fix A+B)  
et avec la fitness IDS-optimale (Fix C) + repair strict + entropy reg (Fix D).

In [ ]:
# ═══════════════════════════════════════════════════════════
# Lancement QIEA v2 sur données OHE corrigées
# ═══════════════════════════════════════════════════════════

# Split interne pour fitness — fit sur train_fix uniquement
X_tr2, X_val2, y_tr2, y_val2 = train_test_split(
    X_train_fix, y_train_fix,
    test_size=0.2, random_state=42, stratify=y_train_fix
)

print(f'📐 Données OHE pour QIEA v2 :')
print(f'   Train : {X_tr2.shape}   Val : {X_val2.shape}')
print(f'   Features totales : {X_train_fix.shape[1]} (après OneHotEncoding)\n')

qiea_v2 = QIEA_IDSOptimal(
    n_features_target = 10,
    n_generations     = 30,
    population_size   = 10,
    rotation_angle    = 0.05 * np.pi,
    n_estimators_eval = 30,
    lambda_fitness    = 0.4,   # 40% F1_macro + 60% Recall_attack
    entropy_reg       = 0.01,  # légère régularisation entropique
    verbose           = True
)

t0_v2 = time.perf_counter()
qiea_v2.fit(X_tr2, y_tr2, X_val2, y_val2, feature_names=FEATURE_NAMES_FIX)
t_v2 = time.perf_counter() - t0_v2

SELECTED_IDX_V2   = qiea_v2.get_selected_indices()
SELECTED_NAMES_V2 = qiea_v2.get_selected_names()
print(f'\n⏱️  Durée QIEA v2 : {t_v2:.1f}s')

## 📊 Comparaison v1 vs v2 : Convergence & Probabilités

On visualise côte à côte les deux versions :
- **v1** : convergence trop rapide, P=1 partout dès la génération 1
- **v2** : convergence progressive grâce à l'entropy regularization,  
  probabilités différenciées → sélection **plus informative**

In [ ]:
# ═══════════════════════════════════════════════════════════
# Visualisation : Convergence v1 vs v2
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('QIEA v1 vs v2 — Convergence & Qualité de la Sélection',
             fontsize=16, fontweight='bold')

gens = range(1, len(qiea.fitness_history) + 1)
gens2 = range(1, len(qiea_v2.fitness_history) + 1)

# ── Graphe 1 : Courbes de fitness comparées ─────────────────
axes[0,0].plot(gens,  qiea.fitness_history,
               color=COLORS['classic'], lw=2.5, label='v1 (Accuracy)',
               marker='o', markersize=4, markevery=3)
axes[0,0].plot(gens2, qiea_v2.fitness_history,
               color=COLORS['quantum'], lw=2.5, label='v2 (λ·F1+Recall_atk)',
               marker='s', markersize=4, markevery=3)
axes[0,0].set_title('Évolution du Fitness', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Génération')
axes[0,0].set_ylabel('Fitness')
axes[0,0].legend(fontsize=10)
axes[0,0].set_ylim([0.85, 1.01])

# ── Graphe 2 : Recall_attack v2 au fil des générations ──────
axes[0,1].plot(gens2, qiea_v2.recall_history,
               color=COLORS['accent'], lw=2.5, label='Recall_attack (v2)',
               marker='D', markersize=4, markevery=3)
axes[0,1].plot(gens2, qiea_v2.f1_history,
               color='#FFC107', lw=2, linestyle='--', label='F1_macro (v2)',
               marker='^', markersize=4, markevery=3)
axes[0,1].axhline(y=rec_bl, color=COLORS['classic'], lw=1.5,
                  linestyle=':', label=f'Baseline Recall={rec_bl:.3f}')
axes[0,1].set_title('Recall_attack & F1_macro (v2)', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Génération')
axes[0,1].set_ylabel('Score')
axes[0,1].legend(fontsize=9)
axes[0,1].set_ylim([0.4, 1.01])

# ── Graphe 3 : Probabilités v1 (P=1 partout = mauvais signe) 
probs_v1 = np.sort(qiea.final_probabilities)[::-1][:20]
axes[1,0].bar(range(len(probs_v1)), probs_v1,
              color=[COLORS['classic'] if p > 0.95 else '#B0BEC5' for p in probs_v1])
axes[1,0].axhline(y=0.5, color='gray', lw=1, linestyle='--', label='Seuil initial 50%')
axes[1,0].set_title('v1 : Probabilités finales sin²(θ)\n⚠️ P=1 partout = convergence agressive',
                    fontsize=11, fontweight='bold')
axes[1,0].set_xlabel('Feature (rank)')
axes[1,0].set_ylabel('P(sélection)')
axes[1,0].set_ylim([0, 1.1])
axes[1,0].legend(fontsize=9)

# ── Graphe 4 : Probabilités v2 (dégradé = meilleure discrimination)
probs_v2 = np.sort(qiea_v2.final_probabilities)[::-1][:20]
bar_colors_v2 = [
    COLORS['quantum'] if p > 0.7 else
    '#A29BFE' if p > 0.4 else '#B0BEC5'
    for p in probs_v2
]
axes[1,1].bar(range(len(probs_v2)), probs_v2, color=bar_colors_v2)
axes[1,1].axhline(y=0.5, color='gray', lw=1, linestyle='--', label='Seuil initial 50%')
axes[1,1].set_title('v2 : Probabilités finales sin²(θ)\n✅ Dégradé = sélection discriminante',
                    fontsize=11, fontweight='bold')
axes[1,1].set_xlabel('Feature (rank)')
axes[1,1].set_ylabel('P(sélection)')
axes[1,1].set_ylim([0, 1.1])
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('05_qiea_v1_vs_v2_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Figure sauvegardée : 05_qiea_v1_vs_v2_convergence.png')

## 🔴 Baseline v2 + 🟢 Modèle QIEA v2 : Évaluation Finale

On réentraîne le RandomForest final sur les données **OHE** (Fix A+B) :
- Baseline v2 : 41 features numériques (sans OHE pour garder la comparaison juste)
- QIEA v2 : 10 features sélectionnées par le QIEA v2 (fitness IDS-optimale)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Évaluation Finale : Baseline v2 vs QIEA v2
# ═══════════════════════════════════════════════════════════

print('═' * 65)
print('  ÉVALUATION FINALE : Baseline v2 vs QIEA v2 (fixes appliqués)')
print('═' * 65)

# ── Baseline v2 (données OHE complètes) ─────────────────────
t0 = time.perf_counter()
rf_bl_v2 = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf_bl_v2.fit(X_train_fix, y_train_fix)
t_bl_v2 = time.perf_counter() - t0

t1 = time.perf_counter()
y_pred_bl_v2 = rf_bl_v2.predict(X_test_fix)
t_infer_bl_v2 = time.perf_counter() - t1

# ── QIEA v2 (10 features OHE sélectionnées) ─────────────────
X_train_qi_v2 = X_train_fix[:, SELECTED_IDX_V2]
X_test_qi_v2  = X_test_fix[:, SELECTED_IDX_V2]

t0 = time.perf_counter()
rf_qi_v2 = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf_qi_v2.fit(X_train_qi_v2, y_train_fix)
t_qi_v2 = time.perf_counter() - t0

t1 = time.perf_counter()
y_pred_qi_v2 = rf_qi_v2.predict(X_test_qi_v2)
t_infer_qi_v2 = time.perf_counter() - t1

# ── Métriques ────────────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    return {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
        'f1_macro' : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_atk': recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    }

m_bl_v2 = compute_metrics(y_test_fix, y_pred_bl_v2)
m_qi_v2 = compute_metrics(y_test_fix, y_pred_qi_v2)

print(f'\n{'Métrique':<20} {'Baseline v2':>15} {'QIEA v2':>15} {'Delta':>10}')
print('─' * 63)
for key in ['accuracy', 'precision', 'recall', 'f1', 'f1_macro', 'recall_atk']:
    delta = m_qi_v2[key] - m_bl_v2[key]
    arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '=')
    print(f'  {key:<18} {m_bl_v2[key]:>14.4f} {m_qi_v2[key]:>14.4f} '
          f'  {arrow}{abs(delta)*100:>5.2f}%')
print('─' * 63)
print(f'  {"Train time":<18} {t_bl_v2:>14.3f}s {t_qi_v2:>13.3f}s '
      f'  ×{t_bl_v2/t_qi_v2:.1f}')
print(f'  {"N features":<18} {X_train_fix.shape[1]:>14} {len(SELECTED_IDX_V2):>14}')

## 🔧 FIX E — Mesure Mémoire Correcte avec `.nbytes`

### Problème
`sys.getsizeof(numpy_array)` retourne la taille de l'objet Python (en-tête),  
**pas** la taille réelle des données en mémoire → affiche ~0 MB pour une vue.

### Solution : `.nbytes`
La propriété `.nbytes` retourne le nombre d'octets **réellement alloués** :
$$\text{nbytes} = \text{n\_rows} \times \text{n\_cols} \times \text{itemsize}$$
Pour un float64 : `itemsize = 8 octets`

In [ ]:
# ═══════════════════════════════════════════════════════════
# FIX E : Mesure Mémoire Correcte avec .nbytes
# ═══════════════════════════════════════════════════════════

import sys

print('═' * 60)
print('  FIX E : Mesure Mémoire — sys.getsizeof vs .nbytes')
print('═' * 60)

print('\n⚠️  AVANT (Fix E) — sys.getsizeof (ERRONÉ) :')
print(f'   Baseline (41 feat) : {sys.getsizeof(X_train)/(1024**2):.4f} MB')
print(f'   QI v1   (10 feat)  : {sys.getsizeof(X_train_qi)/(1024**2):.4f} MB')
print(f'   → Résultat trompeur car sys.getsizeof = taille objet Python,'
      f' pas données numpy')

print('\n✅ APRÈS (Fix E) — .nbytes (CORRECT) :')
arrays = [
    ('Baseline v1  (41 feat, num)',     X_train),
    ('QIEA v1      (10 feat, num)',     X_train_qi),
    ('Baseline v2  (OHE ~122 feat)',    X_train_fix),
    ('QIEA v2      (10 feat, OHE)',     X_train_qi_v2),
]

ref_bytes = X_train.nbytes
print(f'\n   {"Données":<35} {"MB":>8}  {"Économie vs Baseline":>22}')
print('   ' + '─' * 68)
for name, arr in arrays:
    mb   = arr.nbytes / (1024**2)
    econ = (1 - arr.nbytes / ref_bytes) * 100
    flag = '📉' if econ > 0 else ('📈' if econ < 0 else '—')
    print(f'   {name:<35} {mb:>7.2f}  {flag} {econ:>+7.1f}%')

print(f'\n   Formule utilisée : nbytes = n_rows × n_cols × itemsize')
print(f'   Exemple X_train  : {X_train.shape[0]} × {X_train.shape[1]} × '
      f'{X_train.itemsize} octets = {X_train.nbytes/(1024**2):.2f} MB')

## 🏆 Tableau de Bord Final : v1 vs v2 (toutes fixes)

Comparaison complète des **4 versions** :
- `v1_baseline` : RF sur 41 features + LabelEncoder
- `v1_qiea` : QIEA v1 (Accuracy fitness, P=1 partout)
- `v2_baseline` : RF sur données OHE sans leakage
- `v2_qiea` : QIEA v2 (IDS-fitness + repair strict + entropy reg)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Tableau de Bord Final : Toutes Versions
# ═══════════════════════════════════════════════════════════

# Récupération des métriques v1 (depuis le notebook original)
m_bl_v1  = {'recall_atk': rec_bl, 'f1_macro': f1_bl,
             'accuracy': acc_bl,   'recall': rec_bl}
m_qi_v1  = {'recall_atk': rec_qi, 'f1_macro': f1_qi,
             'accuracy': acc_qi,   'recall': rec_qi}

all_results = {
    'v1 Baseline (41, LabelEnc)': {
        **m_bl_v1, 'train_time': t_baseline_train,
        'n_feat': 41,
        'mem_mb': X_train.nbytes/(1024**2)
    },
    'v1 QIEA    (10, Accuracy)': {
        **m_qi_v1, 'train_time': t_qi_train,
        'n_feat': 10,
        'mem_mb': X_train_qi.nbytes/(1024**2)
    },
    'v2 Baseline (OHE, no leak)': {
        **m_bl_v2, 'train_time': t_bl_v2,
        'n_feat': X_train_fix.shape[1],
        'mem_mb': X_train_fix.nbytes/(1024**2)
    },
    'v2 QIEA    (10, IDS-fit)': {
        **m_qi_v2, 'train_time': t_qi_v2,
        'n_feat': 10,
        'mem_mb': X_train_qi_v2.nbytes/(1024**2)
    },
}

# ── Tableau texte ────────────────────────────────────────────
header = f"{'Modèle':<30} {'Acc':>7} {'F1_mac':>8} {'Rec_atk':>9} {'Train(s)':>10} {'N feat':>8} {'Mem(MB)':>9}"
print('═' * len(header))
print(header)
print('─' * len(header))
for name, m in all_results.items():
    print(f"{name:<30} {m['accuracy']:>7.4f} {m['f1_macro']:>8.4f} "
          f"{m['recall_atk']:>9.4f} {m['train_time']:>10.3f} "
          f"{m['n_feat']:>8} {m['mem_mb']:>9.2f}")
print('═' * len(header))

# ── Visualisation ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Tableau de Bord Final — Toutes Versions',
             fontsize=15, fontweight='bold')

names   = list(all_results.keys())
palette = [COLORS['classic'], '#FFA07A', '#A29BFE', COLORS['quantum']]

metrics_to_plot = [
    ('accuracy',   'Accuracy',        axes[0]),
    ('recall_atk', 'Recall_attack',   axes[1]),
    ('f1_macro',   'F1_macro',        axes[2]),
]

for key, label, ax in metrics_to_plot:
    vals = [all_results[n][key] for n in names]
    bars = ax.bar(range(len(names)), vals, color=palette,
                  alpha=0.85, edgecolor='white')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(
        [n.split('(')[0].strip() for n in names],
        rotation=20, ha='right', fontsize=9
    )
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_ylim([min(vals)*0.97, 1.01])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('06_dashboard_all_versions.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Figure sauvegardée : 06_dashboard_all_versions.png')

# ── Résumé exécutif ──────────────────────────────────────────
bl_v1_rec = all_results['v1 Baseline (41, LabelEnc)']['recall_atk']
qi_v2_rec = all_results['v2 QIEA    (10, IDS-fit)']['recall_atk']
bl_v1_mem = all_results['v1 Baseline (41, LabelEnc)']['mem_mb']
qi_v2_mem = all_results['v2 QIEA    (10, IDS-fit)']['mem_mb']

print(f'''
╔══════════════════════════════════════════════════════════╗
║              RÉSUMÉ EXÉCUTIF — FIXES APPLIQUÉS           ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Fix A+B : OneHotEncoder sans leakage                    ║
║  Fix C   : Fitness IDS (λ·F1_macro + Recall_attack)      ║
║  Fix D   : Repair strict + Entropy regularization        ║
║  Fix E   : Mémoire via .nbytes (résultat réel)           ║
║                                                          ║
║  Recall_attack :  {bl_v1_rec:.4f} → {qi_v2_rec:.4f}  """+
f"({'↑' if qi_v2_rec > bl_v1_rec else '↓'}{abs(qi_v2_rec-bl_v1_rec)*100:.2f}% d'attaques détectées en +)"""
║  Mémoire (réelle): {bl_v1_mem:.2f} MB → {qi_v2_mem:.2f} MB ({'↓' if qi_v2_mem < bl_v1_mem else '↑'}{abs(1-qi_v2_mem/bl_v1_mem)*100:.0f}% économisé)
║  N features    :   41 → 10 (-76%)
║                                                          ║
╚══════════════════════════════════════════════════════════╝
''')